# Анализ SFP 40.1 (логика ДР) и тренд автовыявленных факторов

Тетрадка содержит 2 анализа:
1. По логике дефолтности ДР (`build_dataset.ipynb`): сколько компаний ушли в дефолт с фактором `40.1` и какие другие ФП/СФП были у них до дефолта.
2. По списку автовыявляющихся факторов: растет ли их количество по годам.

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", None)

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"
DEF_FILE = f"{DATA_DIR}/default_data.pkl"
AUTO_LIST_FILE = "./autodetected_factors_from_photos.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO_ANALYSIS = pd.Timestamp("2026-12-31")
DATE_TO_DEFAULTS = pd.Timestamp("2025-12-31")
CUTOFF = DATE_TO_ANALYSIS

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}
SEG_ORDER = ["МкБ", "МСБ", "ККБ"]
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)

In [ ]:
# --- Подготовка данных в логике build_dataset.ipynb ---
# CRM
df_crm = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
df_crm.columns = df_crm.columns.str.strip()

df_crm["X_INN"] = df_crm["X_INN"].apply(normalize_inn)
df_crm["IDENTIFICATION_DATE"] = pd.to_datetime(df_crm["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
if "VAL" in df_crm.columns:
    df_crm = df_crm[df_crm["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()

df_crm["X_AREA_RESP"] = df_crm["X_AREA_RESP"].astype(str).str.strip()
df_crm["segment"] = df_crm["X_AREA_RESP"].map(SEGMENT_MAP)
df_crm = df_crm[df_crm["segment"].notna()].copy()
df_crm = df_crm[(df_crm["IDENTIFICATION_DATE"] >= DATE_FROM) & (df_crm["IDENTIFICATION_DATE"] <= DATE_TO_ANALYSIS)].copy()

fp = df_crm[["X_INN", "NUMBER_FP_SFP", "TYPE_FP", "IDENTIFICATION_DATE", "segment"]].copy()
fp = fp.rename(columns={
    "X_INN": "inn",
    "NUMBER_FP_SFP": "fp_type",
    "TYPE_FP": "fp_kind",
    "IDENTIFICATION_DATE": "fp_date",
})
fp = fp.dropna(subset=["inn", "fp_type", "fp_date"]).copy()
fp["fp_type"] = fp["fp_type"].astype(str).str.strip()
fp["fp_kind"] = fp["fp_kind"].astype(str).str.strip().replace({"FP": "ФП", "SFP": "СФП"})

_seg = df_crm[["X_INN", "segment"]].dropna(subset=["X_INN"]).drop_duplicates()
_mode = lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else None
seg_company = (
    _seg.groupby("X_INN").agg(segment=("segment", _mode)).reset_index().rename(columns={"X_INN": "inn"})
)

# Дефолты (как в build_dataset)
df_def = pd.read_pickle(DEF_FILE)
df_def = df_def.astype(str).replace("nan", np.nan)
df_def.columns = df_def.columns.str.strip()
df_def = df_def.rename(columns={"start": "start_date", "cure": "cure_date", "finish": "finish_date"})
df_def["inn"] = df_def["inn"].apply(normalize_inn)
df_def["start_date"] = pd.to_datetime(df_def["start_date"], dayfirst=True, errors="coerce")

date_from = pd.Timestamp("2023-01-01")
date_to = DATE_TO_DEFAULTS
defaults = (
    df_def.dropna(subset=["inn", "start_date"])
    .query("@date_from <= start_date <= @date_to")
    .groupby("inn", as_index=False)
    .agg(default_date=("start_date", "min"))
)
defaults["default_flag"] = 1

companies = fp[["inn"]].drop_duplicates().copy()
companies = companies.merge(defaults[["inn", "default_flag", "default_date"]], on="inn", how="left")
companies["default_flag"] = companies["default_flag"].fillna(0).astype(int)
companies = companies.merge(seg_company[["inn", "segment"]], on="inn", how="left")

# Временной фильтр до дефолта
fp_with_def = fp.merge(companies[["inn", "default_flag", "default_date"]], on="inn", how="inner")
fp_with_def["ref_date"] = fp_with_def["default_date"].fillna(CUTOFF)
fp_filtered = fp_with_def[fp_with_def["fp_date"] < fp_with_def["ref_date"]].copy()
fp_filtered["year"] = fp_filtered["fp_date"].dt.year

print(f"CRM после подготовки: {len(fp):,} событий")
print(f"ФП/СФП до даты дефолта (или cutoff): {len(fp_filtered):,}")
print(f"Компаний в реестре: {len(companies):,}, дефолтных: {int(companies['default_flag'].sum()):,}")

## 1) Дефолтные компании с фактором 40.1 (логика ДР)

In [ ]:
# Когорта дефолтных компаний, у которых до дефолта встречался код 40.1
cohort_40_1 = fp_filtered[(fp_filtered["default_flag"] == 1) & (fp_filtered["fp_type"] == "40.1")].copy()
cohort_inn = cohort_40_1["inn"].dropna().unique()

n_def_total = int(companies["default_flag"].sum())
n_cohort = len(cohort_inn)
print("=" * 72)
print("Когорта дефолтных компаний с фактором 40.1 до дефолта")
print("=" * 72)
print(f"Дефолтных компаний всего: {n_def_total:,}")
print(f"С фактором 40.1 до дефолта: {n_cohort:,} ({(n_cohort / n_def_total * 100 if n_def_total else 0):.2f}%)")

cohort_comp = companies[(companies["inn"].isin(cohort_inn)) & (companies["default_flag"] == 1)].copy()
seg_view = cohort_comp["segment"].value_counts(dropna=False).rename_axis("segment").reset_index(name="Компаний")
display(seg_view)

kind_40_1 = cohort_40_1["fp_kind"].value_counts(dropna=False).rename_axis("fp_kind").reset_index(name="Событий 40.1")
print("Тип события для кода 40.1 в когорте:")
display(kind_40_1)

# ИНН компаний и дата первого появления 40.1
first_40_1 = (
    cohort_40_1.groupby("inn", as_index=False)
    .agg(first_40_1_date=("fp_date", "min"), events_40_1=("fp_date", "size"))
)

cohort_inn_list = (
    cohort_comp[["inn", "segment", "default_date"]]
    .drop_duplicates()
    .merge(first_40_1, on="inn", how="left")
    .sort_values(["first_40_1_date", "inn"])
    .reset_index(drop=True)
)

print("\nИНН компаний с 40.1 и дефолтом:")
display(cohort_inn_list)

In [ ]:
# Какие факторы были до 40.1 и после 40.1 (до даты дефолта)
cohort_events = fp_filtered[(fp_filtered["inn"].isin(cohort_inn)) & (fp_filtered["default_flag"] == 1)].copy()
cohort_events = cohort_events.merge(first_40_1[["inn", "first_40_1_date"]], on="inn", how="left")

# Дедупликация по ключу: ИНН + дата выявления + тип + номер ФП/СФП
dedup_key = ["inn", "fp_date", "fp_kind", "fp_type"]
cohort_events = cohort_events.drop_duplicates(subset=dedup_key).copy()

before_40_1_events = cohort_events[(cohort_events["fp_type"] != "40.1") & (cohort_events["fp_date"] < cohort_events["first_40_1_date"])].copy()
after_40_1_events = cohort_events[(cohort_events["fp_type"] != "40.1") & (cohort_events["fp_date"] > cohort_events["first_40_1_date"])].copy()

# Детальные списки факторов по ИНН (только уникальные события по ключу)
factors_before_40_1 = (
    before_40_1_events[["inn", "segment", "default_date", "first_40_1_date", "fp_date", "fp_kind", "fp_type"]]
    .drop_duplicates(subset=dedup_key)
    .sort_values(["inn", "fp_date", "fp_kind", "fp_type"])
    .reset_index(drop=True)
)

factors_after_40_1 = (
    after_40_1_events[["inn", "segment", "default_date", "first_40_1_date", "fp_date", "fp_kind", "fp_type"]]
    .drop_duplicates(subset=dedup_key)
    .sort_values(["inn", "fp_date", "fp_kind", "fp_type"])
    .reset_index(drop=True)
)

# Агрегаты для проверки гипотезы (на уникальных событиях)
other = cohort_events[cohort_events["fp_type"] != "40.1"].copy()
by_factor = pd.DataFrame(columns=["fp_type", "fp_kind", "Событий", "Уникальных_компаний", "Доля_компаний_в_когорте_%"])
hypothesis_summary = pd.DataFrame(columns=["Показатель", "Значение"])

if len(other) == 0:
    print("Других факторов до даты дефолта в этой когорте не найдено")
else:
    by_factor = (
        other.groupby(["fp_type", "fp_kind"], as_index=False)
        .agg(
            Событий=("inn", "size"),
            Уникальных_компаний=("inn", "nunique"),
        )
        .sort_values(["Уникальных_компаний", "Событий"], ascending=False)
    )
    by_factor["Доля_компаний_в_когорте_%"] = (by_factor["Уникальных_компаний"] / n_cohort * 100).round(2)

    other_per_inn = other.groupby("inn")["fp_type"].nunique()
    n_with_other = int((other_per_inn > 0).sum())
    median_other = float(other_per_inn.median())
    mean_other = float(other_per_inn.mean())

    hypothesis_summary = pd.DataFrame([
        ["Дефолтных компаний с 40.1", n_cohort],
        ["Из них с другими факторами", n_with_other],
        ["Доля с другими факторами, %", round(n_with_other / n_cohort * 100, 2) if n_cohort else 0],
        ["Медиана числа других кодов", round(median_other, 2)],
        ["Среднее число других кодов", round(mean_other, 2)],
    ], columns=["Показатель", "Значение"])

print("\nФакторы до 40.1 (без кода 40.1):")
print(f"Событий: {len(factors_before_40_1):,}")
display(factors_before_40_1)

print("\nФакторы после 40.1 (без кода 40.1, до дефолта):")
print(f"Событий: {len(factors_after_40_1):,}")
display(factors_after_40_1)

print("\nТОП факторов (кроме 40.1) до даты дефолта:")
display(by_factor.head(40))

print("\nПроверка гипотезы:")
display(hypothesis_summary)

## 2) Автовыявляющиеся факторы: динамика по годам и месяцам

In [ ]:
auto_list = pd.read_csv(AUTO_LIST_FILE)
auto_list["factor_code"] = auto_list["factor_code"].astype(str).str.strip()
auto_list["factor_type"] = auto_list["factor_type"].astype(str).str.strip()
auto_factors = auto_list[["factor_type", "factor_code"]].drop_duplicates()

matched_auto = fp_filtered.merge(
    auto_factors,
    left_on=["fp_kind", "fp_type"],
    right_on=["factor_type", "factor_code"],
    how="inner"
)

# Годовая динамика: абсолюты и доля от всех событий года
all_by_year = fp_filtered.groupby("year").size().rename("all_events")
auto_by_year = matched_auto.groupby("year").size().rename("auto_events")
auto_inn_by_year = matched_auto.groupby("year")["inn"].nunique().rename("auto_unique_inn")

yearly = pd.concat([all_by_year, auto_by_year, auto_inn_by_year], axis=1).fillna(0).reset_index()
yearly[["all_events", "auto_events", "auto_unique_inn"]] = yearly[["all_events", "auto_events", "auto_unique_inn"]].astype(int)
yearly["auto_share_%"] = (yearly["auto_events"] / yearly["all_events"] * 100).round(2)

print("Динамика автовыявляющихся факторов по годам:")
display(yearly)

# Помесячная динамика
fp_filtered["year_month"] = fp_filtered["fp_date"].dt.to_period("M").astype(str)
matched_auto["year_month"] = matched_auto["fp_date"].dt.to_period("M").astype(str)

all_by_month = fp_filtered.groupby("year_month").size().rename("all_events")
auto_by_month = matched_auto.groupby("year_month").size().rename("auto_events")
auto_inn_by_month = matched_auto.groupby("year_month")["inn"].nunique().rename("auto_unique_inn")

monthly = pd.concat([all_by_month, auto_by_month, auto_inn_by_month], axis=1).fillna(0).reset_index()
monthly[["all_events", "auto_events", "auto_unique_inn"]] = monthly[["all_events", "auto_events", "auto_unique_inn"]].astype(int)
monthly["auto_share_%"] = (monthly["auto_events"] / monthly["all_events"] * 100).round(2)

print("\nДинамика автовыявляющихся факторов по месяцам:")
display(monthly)

# Помесячно в разрезе ФП/СФП
all_month_kind = fp_filtered.groupby(["year_month", "fp_kind"]).size().rename("all_events").reset_index()
auto_month_kind = matched_auto.groupby(["year_month", "fp_kind"]).size().rename("auto_events").reset_index()
monthly_by_kind = all_month_kind.merge(auto_month_kind, on=["year_month", "fp_kind"], how="left").fillna({"auto_events": 0})
monthly_by_kind["auto_events"] = monthly_by_kind["auto_events"].astype(int)
monthly_by_kind["auto_share_%"] = (monthly_by_kind["auto_events"] / monthly_by_kind["all_events"] * 100).round(2)

print("\nПомесячно по типу фактора (ФП/СФП):")
display(monthly_by_kind)

# Отдельный срез 2026: сколько ФП/СФП всего и сколько из них автовыявляющиеся
all_2026 = fp_filtered[fp_filtered["year"] == 2026].copy()
auto_2026 = matched_auto[matched_auto["year"] == 2026].copy()

all_2026_by_kind = all_2026.groupby("fp_kind").size().rename("all_events_2026")
auto_2026_by_kind = auto_2026.groupby("fp_kind").size().rename("auto_events_2026")

summary_2026 = pd.concat([all_2026_by_kind, auto_2026_by_kind], axis=1).fillna(0).reset_index()
if len(summary_2026) > 0:
    summary_2026[["all_events_2026", "auto_events_2026"]] = summary_2026[["all_events_2026", "auto_events_2026"]].astype(int)
    summary_2026["auto_share_2026_%"] = (summary_2026["auto_events_2026"] / summary_2026["all_events_2026"] * 100).round(2)

print("\nСрез 2026 по типу фактора (ФП/СФП):")
display(summary_2026)

fig, ax1 = plt.subplots(figsize=(10, 4.8))
ax1.plot(yearly["year"], yearly["auto_events"], marker="o", linewidth=2, label="auto_events")
ax1.set_xlabel("Год")
ax1.set_ylabel("Количество событий (автовыявляющиеся)")
ax1.grid(axis="y", alpha=0.25)

ax2 = ax1.twinx()
ax2.plot(yearly["year"], yearly["auto_share_%"], marker="s", linestyle="--", linewidth=1.6, label="auto_share_%")
ax2.set_ylabel("Доля от всех событий, %")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
ax1.set_title("Автовыявляющиеся ФП/СФП: динамика по годам (2023-2026)")
plt.tight_layout()
plt.show()

# Экспорт всех результатов на отдельные листы Excel
out_xlsx = f"{DATA_DIR}/Анализ_SFP40_и_автовыявленных_2023_2026.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    cohort_inn_list.to_excel(writer, sheet_name="sfp40_companies", index=False)
    seg_view.to_excel(writer, sheet_name="sfp40_cohort_segments", index=False)
    kind_40_1.to_excel(writer, sheet_name="sfp40_type_split", index=False)
    factors_before_40_1.to_excel(writer, sheet_name="sfp40_before_40_1", index=False)
    factors_after_40_1.to_excel(writer, sheet_name="sfp40_after_40_1", index=False)
    by_factor.to_excel(writer, sheet_name="sfp40_other_factors", index=False)
    hypothesis_summary.to_excel(writer, sheet_name="sfp40_hypothesis", index=False)

    yearly.to_excel(writer, sheet_name="auto_yearly", index=False)
    monthly.to_excel(writer, sheet_name="auto_monthly", index=False)
    monthly_by_kind.to_excel(writer, sheet_name="auto_monthly_by_type", index=False)
    summary_2026.to_excel(writer, sheet_name="auto_2026_by_type", index=False)

print(f"\nСохранено в Excel: {out_xlsx}")